# Limpieza de datos - Spotify Tracks

Este notebook aplica reglas simples y justificadas para preparar una version limpia del dataset. El archivo original no se modifica.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.clean_data import clean_spotify_data, validate_clean_data

## 1. Cargar los datos originales

In [ ]:
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Spotify_Tracks_Dataset.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'spotify_tracks_clean.csv'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'data_quality' / 'cleaning_summary.csv'

df_raw = pd.read_csv(RAW_PATH)
print(f'Filas originales: {len(df_raw):,}')
df_raw.head()

## 2. Aplicar reglas de limpieza

Las reglas eliminan el indice exportado, registros sin datos esenciales, IDs repetidos y valores no validos de duracion, tempo o compas. Se conserva el primer registro de cada `track_id` para evitar que una misma cancion aparezca varias veces durante el futuro modelamiento.

In [ ]:
df_clean, cleaning_summary = clean_spotify_data(df_raw)
cleaning_summary

## 3. Validar los resultados

In [ ]:
validate_clean_data(df_clean)

quality_check = pd.Series({
    'filas_limpias': len(df_clean),
    'columnas': df_clean.shape[1],
    'valores_nulos': int(df_clean.isna().sum().sum()),
    'track_id_duplicados': int(df_clean['track_id'].duplicated().sum()),
    'generos': int(df_clean['track_genre'].nunique()),
})
quality_check

## 4. Guardar datos procesados y reporte

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(PROCESSED_PATH, index=False)
cleaning_summary.to_csv(REPORT_PATH, index=False)

print(f'Dataset limpio guardado en: {PROCESSED_PATH}')
print(f'Reporte guardado en: {REPORT_PATH}')

## 5. Resultado

El dataset limpio queda sin valores nulos esenciales, sin IDs repetidos y con rangos validos para las variables revisadas. La eliminacion de IDs repetidos evita fuga de informacion, aunque puede reducir la representacion de canciones asociadas a varios generos; esta limitacion debe mencionarse en el informe.